# RL Holdout Evaluation Analysis

Analysis notebook for the independent 100-seed holdout evaluation. It reads `holdout_all_evaluations.csv`, writes statistical result tables, and exports thesis figures.

In [ ]:
from __future__ import annotations

from math import sqrt
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'rl_holdout_evaluation.py').exists():
    BASE_DIR = Path('C:/Workspace ZHAW/6. Semester/BA/BA-Simulation-Optimization/RL')

HOLDOUT_DIR = BASE_DIR / 'results' / 'rl_holdout_results' / 'holdout_100_seeds'
SMOKE_DIR = BASE_DIR / 'results' / 'rl_holdout_results' / 'holdout_smoke_test'
if not (HOLDOUT_DIR / 'holdout_all_evaluations.csv').exists() and (SMOKE_DIR / 'holdout_all_evaluations.csv').exists():
    HOLDOUT_DIR = SMOKE_DIR

ANALYSIS_DIR = BASE_DIR / 'statistical_analysis_results' / 'holdout_100_seeds'
FIGURE_DIR = BASE_DIR.parent.parent / 'Thesis' / 'BA' / 'figures' / 'rl_results'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = HOLDOUT_DIR / 'holdout_all_evaluations.csv'
POLICY_REGISTRY_PATH = HOLDOUT_DIR / 'policy_registry.csv'
print(f'Holdout input: {DATA_PATH}')
print(f'Analysis output: {ANALYSIS_DIR}')
print(f'Figure output: {FIGURE_DIR}')

In [ ]:
df = pd.read_csv(DATA_PATH)
registry = pd.read_csv(POLICY_REGISTRY_PATH) if POLICY_REGISTRY_PATH.exists() else pd.DataFrame()

if 'error' in df.columns:
    df = df[df['error'].fillna('').astype(str).str.len() == 0].copy()

for column in ['total_reward', 'late_order_fraction', 'eval_seed', 'replication']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

policy_labels = df.drop_duplicates('policy_id').set_index('policy_id')['policy_label'].to_dict()
df[['policy_id', 'policy_label', 'replication', 'eval_seed', 'total_reward', 'late_order_fraction']].head()

In [ ]:
ALPHA = 0.05
METRIC = 'total_reward'
TEST_ALTERNATIVE = 'two-sided'

def standard_error(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors='coerce').dropna()
    if len(values) <= 1:
        return float('nan')
    return float(values.std(ddof=1) / sqrt(len(values)))

def describe_reward(values: pd.Series) -> dict[str, float | int]:
    values = pd.to_numeric(values, errors='coerce').dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    return {
        'n': int(len(values)),
        'total_reward_mean': float(values.mean()),
        'total_reward_se': standard_error(values),
        'total_reward_median': float(values.median()),
        'total_reward_q1': float(q1),
        'total_reward_q3': float(q3),
        'total_reward_iqr': float(q3 - q1),
        'total_reward_min': float(values.min()),
        'total_reward_max': float(values.max()),
    }

def holm_bonferroni(p_values: list[float]) -> list[float]:
    m = len(p_values)
    ordered = sorted(enumerate(p_values), key=lambda item: item[1])
    adjusted = [float('nan')] * m
    running_max = 0.0
    for rank, (original_index, p_value) in enumerate(ordered, start=1):
        corrected = min((m - rank + 1) * p_value, 1.0)
        running_max = max(running_max, corrected)
        adjusted[original_index] = running_max
    return adjusted

def paired_wilcoxon(first: pd.Series, second: pd.Series, first_name: str, second_name: str, comparison: str) -> dict[str, object]:
    first = pd.to_numeric(first, errors='coerce')
    second = pd.to_numeric(second, errors='coerce')
    valid = pd.DataFrame({'first': first, 'second': second}).dropna()
    differences = valid['first'] - valid['second']
    try:
        test = stats.wilcoxon(valid['first'], valid['second'], alternative=TEST_ALTERNATIVE)
        statistic = float(test.statistic)
        p_value = float(test.pvalue)
    except ValueError:
        statistic = 0.0
        p_value = 1.0
    return {
        'comparison': comparison,
        'metric': METRIC,
        'first_group': first_name,
        'second_group': second_name,
        'n_pairs': int(len(valid)),
        'mean_first': float(valid['first'].mean()),
        'mean_second': float(valid['second'].mean()),
        'standard_error_first': standard_error(valid['first']),
        'standard_error_second': standard_error(valid['second']),
        'median_first': float(valid['first'].median()),
        'median_second': float(valid['second'].median()),
        'q1_first': float(valid['first'].quantile(0.25)),
        'q1_second': float(valid['second'].quantile(0.25)),
        'q3_first': float(valid['first'].quantile(0.75)),
        'q3_second': float(valid['second'].quantile(0.75)),
        'iqr_first': float(valid['first'].quantile(0.75) - valid['first'].quantile(0.25)),
        'iqr_second': float(valid['second'].quantile(0.75) - valid['second'].quantile(0.25)),
        'mean_difference_first_minus_second': float(differences.mean()),
        'standard_error_difference': standard_error(differences),
        'median_difference_first_minus_second': float(differences.median()),
        'test': 'Wilcoxon signed-rank',
        'alternative': TEST_ALTERNATIVE,
        'test_statistic': statistic,
        'p_value': p_value,
        'alpha': ALPHA,
        'significant': bool(p_value < ALPHA),
    }

def paired_policy_test(first_policy: str, second_policy: str, comparison: str) -> dict[str, object] | None:
    available = set(df['policy_id'])
    if first_policy not in available or second_policy not in available:
        return None
    paired = df[df['policy_id'].isin([first_policy, second_policy])].pivot_table(
        index='eval_seed', columns='policy_id', values=METRIC, aggfunc='first'
    ).dropna()
    if paired.empty:
        return None
    return paired_wilcoxon(
        paired[first_policy],
        paired[second_policy],
        policy_labels.get(first_policy, first_policy),
        policy_labels.get(second_policy, second_policy),
        comparison,
    )

def save_tests(rows: list[dict[str, object]], path: Path, holm: bool = False) -> pd.DataFrame:
    out = pd.DataFrame(rows)
    if holm and not out.empty:
        out['p_value_holm'] = holm_bonferroni(out['p_value'].astype(float).tolist())
        out['significant_holm'] = out['p_value_holm'] < ALPHA
    out.to_csv(path, index=False)
    return out

In [ ]:
descriptive_rows = []
for policy_id, group in df.groupby('policy_id'):
    row = {
        'policy_id': policy_id,
        'policy_label': policy_labels.get(policy_id, policy_id),
        'policy_group': group['policy_group'].iloc[0] if 'policy_group' in group else '',
    }
    row.update(describe_reward(group['total_reward']))
    late = pd.to_numeric(group['late_order_fraction'], errors='coerce').dropna()
    row.update({
        'late_order_fraction_mean': float(late.mean()),
        'late_order_fraction_se': standard_error(late),
        'late_order_fraction_median': float(late.median()),
    })
    descriptive_rows.append(row)

descriptives = pd.DataFrame(descriptive_rows).sort_values('total_reward_mean', ascending=False)
descriptives.to_csv(ANALYSIS_DIR / 'holdout_policy_descriptives.csv', index=False)
descriptives

In [ ]:
h4_row = paired_policy_test(
    'bo_tuned_fixed758',
    'manual_fixed758',
    'H4 holdout: BO-tuned fixed758 vs manual fixed758',
)
h4 = save_tests(
    [] if h4_row is None else [h4_row],
    ANALYSIS_DIR / 'holdout_h4_tuned_fixed758_vs_manual_fixed758.csv',
)

reference_policies = [
    'baseline_fifo',
    'baseline_earliest_due_date',
    'baseline_longest_waiting_time',
    'baseline_highest_lateness_risk',
    'random_agent',
    'manual_fixed758',
]
rows = []
for reference in reference_policies:
    row = paired_policy_test('bo_tuned_fixed758', reference, f'BO-tuned fixed758 vs {reference}')
    if row is not None:
        rows.append(row)
tuned_vs_references = save_tests(rows, ANALYSIS_DIR / 'holdout_tuned_fixed758_vs_references_tests.csv', holm=True)

rows = []
for reference in reference_policies[:-1]:
    row = paired_policy_test('manual_fixed758', reference, f'Manual fixed758 vs {reference}')
    if row is not None:
        rows.append(row)
manual_vs_references = save_tests(rows, ANALYSIS_DIR / 'holdout_manual_fixed758_vs_references_tests.csv', holm=True)

display(h4)
display(tuned_vs_references)
display(manual_vs_references)

In [ ]:
tuned_variants = [
    'bo_tuned_joint100',
    'bo_tuned_fixed120',
    'bo_tuned_fixed758',
    'bo_tuned_gamma_high',
    'bo_tuned_gamma_low',
]
tuning_variant_descriptives = descriptives[descriptives['policy_id'].isin(tuned_variants)].copy()
tuning_variant_descriptives.to_csv(ANALYSIS_DIR / 'holdout_tuning_variant_descriptives.csv', index=False)

rows = []
for variant in [policy for policy in tuned_variants if policy != 'bo_tuned_fixed758']:
    row = paired_policy_test('bo_tuned_fixed758', variant, f'BO-tuned fixed758 vs {variant}')
    if row is not None:
        rows.append(row)
tuned_variant_tests = save_tests(
    rows,
    ANALYSIS_DIR / 'holdout_tuned_fixed758_vs_other_tuned_variants_tests.csv',
    holm=True,
)

display(tuning_variant_descriptives)
display(tuned_variant_tests)

In [ ]:
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def label_for(policy_id: str, width: int = 18) -> str:
    return '\n'.join(textwrap.wrap(policy_labels.get(policy_id, policy_id), width=width))

def save_figure(fig: plt.Figure, stem: str) -> None:
    for suffix in ['pdf', 'png']:
        fig.savefig(FIGURE_DIR / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()

def ordered_available(policy_ids: list[str], sort_by_mean: bool = False) -> list[str]:
    available = [policy for policy in policy_ids if policy in set(df['policy_id'])]
    if sort_by_mean:
        means = df[df['policy_id'].isin(available)].groupby('policy_id')['total_reward'].mean()
        available = sorted(available, key=lambda policy: means.loc[policy], reverse=True)
    return available

def box_strip(policy_ids: list[str], stem: str, title: str, metric: str = 'total_reward', sort_by_mean: bool = False) -> None:
    policies = ordered_available(policy_ids, sort_by_mean=sort_by_mean)
    if len(policies) < 1:
        print(f'Skipping {stem}: no requested policies available.')
        return
    values = [df.loc[df['policy_id'] == policy, metric].dropna().to_numpy() for policy in policies]
    fig_width = max(5.5, 1.15 * len(policies))
    fig, ax = plt.subplots(figsize=(fig_width, 4.2))
    box = ax.boxplot(values, patch_artist=True, showfliers=False, widths=0.55)
    for patch in box['boxes']:
        patch.set_facecolor('#d9e6f2')
        patch.set_edgecolor('#2f4858')
    rng = np.random.default_rng(123)
    for idx, series in enumerate(values, start=1):
        jitter = rng.normal(0, 0.045, size=len(series))
        ax.scatter(np.full(len(series), idx) + jitter, series, s=16, alpha=0.55, color='#c44e52', linewidth=0)
    ax.set_xticks(range(1, len(policies) + 1))
    ax.set_xticklabels([label_for(policy) for policy in policies], rotation=0, ha='center')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    save_figure(fig, stem)

final_policy_order = [
    'bo_tuned_fixed758',
    'baseline_highest_lateness_risk',
    'manual_fixed758',
    'baseline_fifo',
    'baseline_longest_waiting_time',
    'random_agent',
    'baseline_earliest_due_date',
]

box_strip(
    ['bo_tuned_fixed758', 'manual_fixed758'],
    'rl_holdout_tuned_vs_manual_reward_distribution',
    'Holdout reward distribution: tuned vs manual fixed758',
)
box_strip(
    final_policy_order,
    'rl_holdout_final_policy_reward_distribution',
    'Holdout reward distribution: final policies',
    sort_by_mean=True,
)
box_strip(
    tuned_variants,
    'rl_holdout_tuning_variant_reward_distribution',
    'Holdout reward distribution: BO tuning variants',
    sort_by_mean=True,
)
box_strip(
    tuned_variants,
    'rl_holdout_tuning_variant_late_fraction',
    'Holdout late fraction: BO tuning variants',
    metric='late_order_fraction',
    sort_by_mean=False,
)

In [ ]:
summary_policies = ordered_available(final_policy_order, sort_by_mean=True)
if summary_policies:
    summary = df[df['policy_id'].isin(summary_policies)].groupby('policy_id').agg(
        reward_mean=('total_reward', 'mean'),
        reward_se=('total_reward', standard_error),
        late_mean=('late_order_fraction', 'mean'),
        late_se=('late_order_fraction', standard_error),
    ).loc[summary_policies]

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharex=False)
    x = np.arange(len(summary))
    labels = [label_for(policy, width=16) for policy in summary.index]

    axes[0].bar(x, summary['reward_mean'], yerr=summary['reward_se'], color='#4c78a8', capsize=3)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=35, ha='right')
    axes[0].set_ylabel('Mean Total Reward')
    axes[0].grid(axis='y', alpha=0.25)

    axes[1].bar(x, summary['late_mean'], yerr=summary['late_se'], color='#f58518', capsize=3)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=35, ha='right')
    axes[1].set_ylabel('Mean Late Fraction')
    axes[1].grid(axis='y', alpha=0.25)

    fig.suptitle('Holdout summary: reward and late fraction')
    fig.tight_layout()
    save_figure(fig, 'rl_holdout_final_policy_summary')
else:
    print('Skipping summary figure: no final policies available.')